# Clasificación semántica de relevancia de noticias

## Introducción

Este notebook forma parte del desarrollo del prototipo *Communication Intelligence Assistant*, orientado al monitoreo y análisis de información proveniente de medios digitales mediante técnicas de Procesamiento del Lenguaje Natural (PLN).

En esta etapa se estudiará la utilización de un modelo Transformer multilingüe, basado en mDeBERTa y ajustado para inferencia de lenguaje natural (NLI), con el objetivo de determinar la relevancia temática de noticias previamente recopiladas y normalizadas.

La clasificación *zero-shot* permite evaluar la relación semántica entre una noticia y una descripción temática sin requerir un entrenamiento supervisado específico para cada tema de vigilancia.

## Objetivos

- Recuperar noticias normalizadas desde PostgreSQL.
- Preparar los textos para su clasificación semántica.
- Implementar una primera clasificación mediante una hipótesis temática general.
- Explorar una estrategia alternativa basada en múltiples hipótesis específicas.
- Comparar los resultados preliminares de ambas estrategias.

## Alcance

Esta primera experimentación tiene carácter exploratorio. La evaluación cuantitativa definitiva requerirá un conjunto de referencia etiquetado manualmente y métricas como Precision, Recall y F1-score.

El procesamiento se realizará localmente, manteniendo el enfoque *on-premise* del proyecto.

In [8]:
import pandas as pd
import psycopg
import os
from dotenv import load_dotenv

In [5]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

True

In [6]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [9]:
with get_database_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database_name = cursor.fetchone()[0]

print(f"Conectado correctamente a la base de datos: {database_name}")

Conectado correctamente a la base de datos: cia_db


## Recuperación de noticias candidatas

Se recuperan desde PostgreSQL las noticias que presentan al menos
una coincidencia con las palabras clave configuradas.

Este conjunto constituye la entrada inicial del clasificador
semántico. La selección mediante palabras clave representa una
etapa de recuperación de candidatos, no una validación definitiva
de relevancia temática.

Para la primera experimentación se utilizarán el título y la
descripción de cada noticia.

In [10]:
def get_candidate_news():
    """
    Recupera noticias asociadas a palabras clave activas.

    Returns:
        pd.DataFrame: Noticias candidatas con sus identificadores,
        títulos, descripciones y fechas de publicación.
    """
    query = """
        SELECT
            rn.rss_news_id,
            rn.title,
            rn.description,
            rn.published_at
        FROM rss_news AS rn
        WHERE EXISTS (
            SELECT 1
            FROM rss_news_keywords AS rnk
            INNER JOIN keywords AS k
                ON rnk.fk_keyword_id = k.keyword_id
            WHERE rnk.fk_rss_news_id = rn.rss_news_id
                AND k.is_active = TRUE
        )
        ORDER BY rn.published_at DESC, rn.rss_news_id DESC;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            records = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(records, columns=columns)


candidate_news = get_candidate_news()

print(f"Noticias candidatas recuperadas: {len(candidate_news)}")

display(candidate_news.head(10))

Noticias candidatas recuperadas: 70


,rss_news_id,title,description,published_at
0,3,Bill Gates advierte sobre la IA: “será el mayo...,"Según el fundador de Microsoft, esta tecnologí...",2026-08-27 13:11:37-03:00
1,7,Time reveló las 100 personalidades más influye...,Veronyka Gimenes es la única latinoamericana q...,2026-08-27 12:37:49-03:00
2,8,Un informe reveló cómo 700 agentes de IA de Op...,"La empresa, conocida por crear ChatGPT, recono...",2026-08-27 12:30:29-03:00
3,12,El costo de Bitcoin para este jueves,La criptomoneda más popular del mercado de act...,2026-08-27 11:32:46-03:00
4,13,Bill Gates pide reservar empleos solo para hum...,"En un ensayo de casi 6.000 palabras, el cofund...",2026-08-27 11:17:32-03:00
5,16,Cada vez más países ven publicidad en ChatGPT:...,Solo las cuentas de usuarios mayores de 18 año...,2026-08-27 11:11:23-03:00
6,17,Los centros de datos de Estados Unidos ya cons...,Las instalaciones tecnológicas consumieron cer...,2026-08-27 11:03:44-03:00
7,19,Propinas para robots: la polémica detrás de la...,La automatización en coctelería incorpora term...,2026-08-27 10:29:10-03:00
8,21,Amazon se une a Nvidia y tendrá dos millones d...,El gigante tecnológico desplegará los nuevos p...,2026-08-27 10:19:49-03:00
9,24,Un robot humanoide supera el récord humano de ...,El mismo dispositivo logró completar una carre...,2026-08-27 09:39:07-03:00


## Preparación del texto para clasificación

Para representar cada noticia se combinarán su título y descripción.

Esta estrategia permite incorporar información contextual adicional
al título, manteniendo una entrada relativamente breve para el
modelo Transformer.

No se aplicarán técnicas tradicionales como stemming, lematización
o eliminación de stopwords, dado que los modelos Transformer
utilizan el contexto lingüístico y su propio tokenizador.

In [11]:
def build_news_text(title, description):
    """
    Construye el texto de entrada para la clasificación semántica.

    Args:
        title (str): Título de la noticia.
        description (str): Descripción de la noticia.

    Returns:
        str: Texto normalizado que combina título y descripción.
    """
    title = title if pd.notna(title) else ""
    description = description if pd.notna(description) else ""

    text_parts = [
        f"Título: {title.strip()}",
        f"Descripción: {description.strip()}",
    ]

    return "\n".join(text_parts)


candidate_news["classification_text"] = candidate_news.apply(
    lambda row: build_news_text(
        title=row["title"],
        description=row["description"],
    ),
    axis=1,
)

display(
    candidate_news[
        ["rss_news_id", "classification_text"]
    ].head(5)
)

,rss_news_id,classification_text
0,3,Título: Bill Gates advierte sobre la IA: “será...
1,7,Título: Time reveló las 100 personalidades más...
2,8,Título: Un informe reveló cómo 700 agentes de ...
3,12,Título: El costo de Bitcoin para este jueves\n...
4,13,Título: Bill Gates pide reservar empleos solo ...


## Carga del modelo de clasificación zero-shot

Se utilizará el modelo `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`,
basado en la arquitectura mDeBERTa y ajustado para tareas de
inferencia de lenguaje natural (NLI).

El modelo permite evaluar la compatibilidad semántica entre
un texto y una hipótesis formulada en lenguaje natural.

Se utilizará la biblioteca Transformers de Hugging Face para
cargar el modelo y su tokenizador.

Los archivos del modelo se descargarán inicialmente y quedarán
almacenados en la caché local para posteriores ejecuciones.
La inferencia podrá realizarse sin enviar las noticias a
servicios externos.

In [12]:
import torch
import transformers

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")

if torch.cuda.is_available():
    device = 0
    print(f"Dispositivo: GPU - {torch.cuda.get_device_name(0)}")
else:
    device = -1
    print("Dispositivo: CPU")

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cpu
Transformers: 5.20.2
Dispositivo: CPU


In [13]:
from transformers import pipeline


MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"


def load_relevance_classifier(model_name, device=-1):
    """
    Carga un modelo Transformer para clasificación zero-shot.

    Args:
        model_name (str): Identificador del modelo en Hugging Face.
        device (int): Dispositivo de inferencia.
            -1 indica CPU y 0 indica la primera GPU CUDA.

    Returns:
        Pipeline: Clasificador zero-shot inicializado.
    """
    classifier = pipeline(
        task="zero-shot-classification",
        model=model_name,
        tokenizer=model_name,
        device=device,
    )

    return classifier


relevance_classifier = load_relevance_classifier(
    model_name=MODEL_NAME,
    device=device,
)

print("Modelo de clasificación cargado correctamente.")

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\enzor\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\Trabajo final UGR\communication-intelligence-assistant\.venv\L

Modelo de clasificación cargado correctamente.


In [19]:
model = relevance_classifier.model

print("Etiquetas del modelo:")
print(model.config.id2label)

print("\nCorrespondencia inversa:")
print(model.config.label2id)

print("\nTipo de datos de los parámetros:")
print(next(model.parameters()).dtype)

print("\nDispositivo:")
print(next(model.parameters()).device)

Etiquetas del modelo:
{0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Correspondencia inversa:
{'contradiction': 2, 'entailment': 0, 'neutral': 1}

Tipo de datos de los parámetros:
torch.float16

Dispositivo:
cpu


In [20]:
# Convertir los parámetros del modelo a precisión FP32.
relevance_classifier.model.float()

# Verificar la precisión utilizada.
model_dtype = next(
    relevance_classifier.model.parameters()
).dtype

print(f"Precisión actual del modelo: {model_dtype}")

Precisión actual del modelo: torch.float32


In [23]:
# Inspeccionar los textos utilizados en la clasificación.
for _, news in sample_news.iterrows():
    print(f"\n{'=' * 80}")
    print(f"Noticia ID: {news['rss_news_id']}")
    print(f"\n{news['classification_text']}")
    print(f"{'=' * 80}")


Noticia ID: 3

Título: Bill Gates advierte sobre la IA: “será el mayor igualador jamás inventado o la peor fuente de injusticia”
Descripción: Según el fundador de Microsoft, esta tecnología genera riesgos como la destrucción de empleos, el aumento de la desinformación y el debilitamiento del pensamiento crítico

Noticia ID: 7

Título: Time reveló las 100 personalidades más influyentes de la IA: quién es la única latina y el único español en la lista
Descripción: Veronyka Gimenes es la única latinoamericana que aparece en la lista. Es cofundadora de una organización que trabaja para garantizar la diversidad en la tecnología

Noticia ID: 8

Título: Un informe reveló cómo 700 agentes de IA de OpenAI se coordinaron para atacar Hugging Face y engañar a los sistemas de evaluación en un ataque secreto
Descripción: La empresa, conocida por crear ChatGPT, reconoció además que desde fines de mayo un equipo interno ya había detectado señales tempranas de comunicación no autorizada y acceso indeb

## Primera inferencia: clasificación mediante una hipótesis general

En esta etapa se realizará una primera prueba exploratoria del modelo
mDeBERTa sobre cinco noticias candidatas.

Se evaluará la compatibilidad semántica entre el título y la descripción
de cada noticia y una hipótesis general relacionada con inteligencia
artificial.

Para ello se utilizará una clasificación zero-shot basada en inferencia
de lenguaje natural (NLI), sin entrenamiento adicional.

Se compararán las hipótesis de relevancia y no relevancia, utilizando
las puntuaciones del modelo para analizar preliminarmente su capacidad
de discriminación temática.

Los resultados no constituyen todavía una evaluación cuantitativa,
ya que no se dispone de etiquetas de referencia.

In [14]:
TOPIC_DESCRIPTION = (
    "inteligencia artificial, aprendizaje automático, "
    "modelos generativos y sus aplicaciones"
)

RELEVANCE_LABELS = [
    "relevante",
    "no relevante",
]

HYPOTHESIS_TEMPLATE = (
    "Esta noticia sobre "
    f"{TOPIC_DESCRIPTION} "
    "es {}."
)


def classify_news_relevance(text, classifier):
    """
    Clasifica la relevancia temática de una noticia mediante zero-shot.

    Args:
        text (str): Texto de la noticia.
        classifier: Pipeline de clasificación zero-shot.

    Returns:
        dict: Etiqueta predicha y puntuaciones de clasificación.
    """
    result = classifier(
        sequences=text,
        candidate_labels=RELEVANCE_LABELS,
        hypothesis_template=HYPOTHESIS_TEMPLATE,
        multi_label=False,
        truncation=True,
    )

    scores = dict(zip(result["labels"], result["scores"]))

    return {
        "predicted_label": result["labels"][0],
        "relevance_score": scores["relevante"],
        "non_relevance_score": scores["no relevante"],
    }

In [15]:
sample_news = candidate_news.head(5).copy()

classification_results = []

for _, news in sample_news.iterrows():
    prediction = classify_news_relevance(
        text=news["classification_text"],
        classifier=relevance_classifier,
    )

    classification_results.append({
        "rss_news_id": news["rss_news_id"],
        "title": news["title"],
        **prediction,
    })

df_classification_results = pd.DataFrame(classification_results)

display(df_classification_results)

,rss_news_id,title,predicted_label,relevance_score,non_relevance_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,relevante,0.983463,0.016537
1,7,Time reveló las 100 personalidades más influye...,relevante,0.817575,0.182426
2,8,Un informe reveló cómo 700 agentes de IA de Op...,relevante,0.931213,0.068787
3,12,El costo de Bitcoin para este jueves,relevante,0.912358,0.087642
4,13,Bill Gates pide reservar empleos solo para hum...,relevante,0.858481,0.141519


## Experimento 2: reformulación de la hipótesis temática

Se realizará una segunda prueba de clasificación zero-shot utilizando
una hipótesis orientada directamente a la temática de vigilancia.

A diferencia del experimento anterior, se evaluará si la noticia
trata principalmente sobre inteligencia artificial, en lugar de
utilizar las etiquetas abstractas de relevancia.

Se mantendrá el mismo conjunto de cinco noticias para facilitar
la comparación exploratoria de resultados.

In [16]:
TOPIC_LABEL = "inteligencia artificial"

TOPIC_HYPOTHESIS_TEMPLATE = (
    "El tema principal de esta noticia es {}."
)

TOPIC_LABELS = [
    TOPIC_LABEL,
    "un tema diferente de inteligencia artificial",
]


def classify_news_topic(text, classifier):
    """
    Evalúa la temática principal de una noticia mediante zero-shot.

    Args:
        text (str): Título y descripción de la noticia.
        classifier: Pipeline de clasificación zero-shot.

    Returns:
        dict: Etiqueta temática predicha y puntuaciones del modelo.
    """
    result = classifier(
        sequences=text,
        candidate_labels=TOPIC_LABELS,
        hypothesis_template=TOPIC_HYPOTHESIS_TEMPLATE,
        multi_label=False,
        truncation=True,
    )

    scores = dict(zip(result["labels"], result["scores"]))

    return {
        "predicted_topic": result["labels"][0],
        "topic_score": scores[TOPIC_LABEL],
    }


topic_results = []

for _, news in sample_news.iterrows():
    prediction = classify_news_topic(
        text=news["classification_text"],
        classifier=relevance_classifier,
    )

    topic_results.append({
        "rss_news_id": news["rss_news_id"],
        "title": news["title"],
        **prediction,
    })

df_topic_results = pd.DataFrame(topic_results)

display(df_topic_results)

,rss_news_id,title,predicted_topic,topic_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,inteligencia artificial,0.986585
1,7,Time reveló las 100 personalidades más influye...,inteligencia artificial,0.843831
2,8,Un informe reveló cómo 700 agentes de IA de Op...,inteligencia artificial,0.876951
3,12,El costo de Bitcoin para este jueves,inteligencia artificial,0.507812
4,13,Bill Gates pide reservar empleos solo para hum...,inteligencia artificial,0.759047


## Experimento 3: evaluación directa mediante inferencia de lenguaje natural

Se evaluará la relación entre el contenido de cada noticia y una
hipótesis temática mediante inferencia de lenguaje natural (NLI).

El modelo distinguirá tres posibles relaciones:

- Entailment: el contenido respalda la hipótesis.
- Neutral: el contenido no permite confirmar ni refutar la hipótesis.
- Contradiction: el contenido contradice la hipótesis.

Este enfoque permitirá examinar las puntuaciones originales del
modelo NLI sin normalizarlas entre etiquetas temáticas alternativas.

La puntuación de entailment se utilizará como indicador exploratorio
de compatibilidad temática, sin considerarla una probabilidad
calibrada de relevancia.

In [21]:
import torch
import pandas as pd


TOPIC_HYPOTHESIS = (
    "Esta noticia trata principalmente sobre inteligencia artificial."
)


def evaluate_topic_entailment(text, hypothesis, classifier):
    """
    Evalúa la relación NLI entre una noticia y una hipótesis temática.

    Args:
        text (str): Texto de la noticia.
        hypothesis (str): Afirmación temática que se desea evaluar.
        classifier: Pipeline zero-shot previamente inicializado.

    Returns:
        dict: Puntuaciones NLI de entailment, neutral y contradiction.
    """
    tokenizer = classifier.tokenizer
    model = classifier.model

    inputs = tokenizer(
        text,
        hypothesis,
        return_tensors="pt",
        truncation="only_first",
        max_length=512,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1,
    )[0]

    label_scores = {
        model.config.id2label[index].lower(): score.item()
        for index, score in enumerate(probabilities)
    }

    return {
        "entailment_score": label_scores["entailment"],
        "neutral_score": label_scores["neutral"],
        "contradiction_score": label_scores["contradiction"],
    }


nli_results = []

for _, news in sample_news.iterrows():
    prediction = evaluate_topic_entailment(
        text=news["classification_text"],
        hypothesis=TOPIC_HYPOTHESIS,
        classifier=relevance_classifier,
    )

    nli_results.append({
        "rss_news_id": news["rss_news_id"],
        "title": news["title"],
        **prediction,
    })

df_nli_results = pd.DataFrame(nli_results)

display(df_nli_results)

,rss_news_id,title,entailment_score,neutral_score,contradiction_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,0.001043,0.998777,0.000180
1,7,Time reveló las 100 personalidades más influye...,0.000492,0.998938,0.000569
2,8,Un informe reveló cómo 700 agentes de IA de Op...,0.000338,0.999428,0.000235
3,12,El costo de Bitcoin para este jueves,0.000415,0.394661,0.604924
4,13,Bill Gates pide reservar empleos solo para hum...,0.000416,0.999237,0.000347


## Experimento 4: hipótesis temática simplificada

Se evaluará una hipótesis más sencilla que no exige determinar
si la inteligencia artificial constituye el tema principal.

El objetivo es analizar si una formulación menos restrictiva
permite identificar correctamente las noticias relacionadas
con la temática de vigilancia.

Se conservarán los mismos cinco textos para mantener
la comparabilidad entre experimentos.

In [24]:
SIMPLE_TOPIC_HYPOTHESIS = (
    "La noticia habla sobre inteligencia artificial."
)

simple_nli_results = []

for _, news in sample_news.iterrows():
    prediction = evaluate_topic_entailment(
        text=news["classification_text"],
        hypothesis=SIMPLE_TOPIC_HYPOTHESIS,
        classifier=relevance_classifier,
    )

    simple_nli_results.append({
        "rss_news_id": news["rss_news_id"],
        "title": news["title"],
        **prediction,
    })

df_simple_nli_results = pd.DataFrame(simple_nli_results)

display(df_simple_nli_results)

,rss_news_id,title,entailment_score,neutral_score,contradiction_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,0.998671,0.001206,0.000123
1,7,Time reveló las 100 personalidades más influye...,0.997252,0.002625,0.000123
2,8,Un informe reveló cómo 700 agentes de IA de Op...,0.996118,0.003819,0.000063
3,12,El costo de Bitcoin para este jueves,0.000291,0.002454,0.997254
4,13,Bill Gates pide reservar empleos solo para hum...,0.997712,0.002174,0.000114


## Construcción del conjunto de evaluación

Se construirá un conjunto de datos etiquetado manualmente para
evaluar el desempeño del clasificador de relevancia temática.

Cada noticia será revisada a partir de su título y descripción,
utilizando los siguientes criterios:

- 1 (relevante): la inteligencia artificial constituye un tema
  sustancial de la noticia.
- 0 (no relevante): la inteligencia artificial no constituye
  un tema sustancial o aparece únicamente como una mención incidental.
- -1 (dudosa): la información disponible es insuficiente o ambigua.

Las etiquetas humanas se mantendrán independientes de las
predicciones del modelo para evitar sesgos durante la evaluación.

Posteriormente, las etiquetas definitivas se utilizarán como
referencia para calcular precision, recall y F1-score.

In [25]:
# Mostrar todas las noticias candidatas para revisión manual.
for _, news in candidate_news.iterrows():
    print(f"ID: {news['rss_news_id']}")
    print(f"Título: {news['title']}")
    print(f"Descripción: {news['description']}")
    print("-" * 80)

ID: 3
Título: Bill Gates advierte sobre la IA: “será el mayor igualador jamás inventado o la peor fuente de injusticia”
Descripción: Según el fundador de Microsoft, esta tecnología genera riesgos como la destrucción de empleos, el aumento de la desinformación y el debilitamiento del pensamiento crítico
--------------------------------------------------------------------------------
ID: 7
Título: Time reveló las 100 personalidades más influyentes de la IA: quién es la única latina y el único español en la lista
Descripción: Veronyka Gimenes es la única latinoamericana que aparece en la lista. Es cofundadora de una organización que trabaja para garantizar la diversidad en la tecnología
--------------------------------------------------------------------------------
ID: 8
Título: Un informe reveló cómo 700 agentes de IA de OpenAI se coordinaron para atacar Hugging Face y engañar a los sistemas de evaluación en un ataque secreto
Descripción: La empresa, conocida por crear ChatGPT, reconoci

In [26]:
# Etiquetas preliminares asignadas mediante revisión asistida.
relevant_ids = [
    3, 7, 8, 13, 16, 17, 27, 30, 39, 48,
    49, 52, 66, 71, 80, 82, 85, 89, 96,
    109, 114, 116, 117, 119, 122, 125,
    136, 148, 149, 154, 166,
]

uncertain_ids = [
    21, 24, 35, 93, 94, 95, 97, 111, 113, 135,
]

not_relevant_ids = [
    12, 19, 32, 36, 42, 46, 56, 57, 62, 69,
    76, 84, 98, 112, 115, 118, 120, 121,
    123, 124, 126, 127, 128, 129, 130,
    131, 132, 133, 134,
]

# Construir el diccionario de etiquetas.
manual_labels = {
    **dict.fromkeys(relevant_ids, 1),
    **dict.fromkeys(not_relevant_ids, 0),
    **dict.fromkeys(uncertain_ids, -1),
}

# Crear el conjunto de evaluación sin modificar el original.
evaluation_dataset = candidate_news.copy()

evaluation_dataset["manual_label"] = (
    evaluation_dataset["rss_news_id"].map(manual_labels)
)

# Validar que todas las noticias tengan una etiqueta.
assert evaluation_dataset["manual_label"].notna().all()
assert len(evaluation_dataset) == 70

print("Distribución de etiquetas:")
print(evaluation_dataset["manual_label"].value_counts())

display(
    evaluation_dataset[
        ["rss_news_id", "title", "manual_label"]
    ]
)

Distribución de etiquetas:
manual_label
 1    31
 0    29
-1    10
Name: count, dtype: int64


,rss_news_id,title,manual_label
0,3,Bill Gates advierte sobre la IA: “será el mayo...,1
1,7,Time reveló las 100 personalidades más influye...,1
2,8,Un informe reveló cómo 700 agentes de IA de Op...,1
3,12,El costo de Bitcoin para este jueves,0
4,13,Bill Gates pide reservar empleos solo para hum...,1
...,...,...,...
65,122,¿De qué manera las universidades promueven el ...,1
66,154,Memoria Temporal Jerárquica Bioinspirada para ...,1
67,135,¿Cree que su jefe es frío e insensible? Espere...,-1
68,166,El futuro de la IA y su impacto en el conocimi...,1


In [27]:
def get_remaining_news():
    """
    Recupera las noticias que no fueron seleccionadas
    por el filtro inicial de palabras clave.

    Returns:
        pd.DataFrame: Noticias pendientes de evaluación.
    """
    query = """
        SELECT
            n.rss_news_id,
            n.title,
            n.description
        FROM rss_news AS n
        WHERE NOT EXISTS (
            SELECT 1
            FROM rss_news_keywords AS nk
            INNER JOIN keywords AS k
                ON nk.fk_keyword_id = k.keyword_id
            WHERE nk.fk_rss_news_id = n.rss_news_id
                AND k.is_active = TRUE
        )
        ORDER BY n.rss_news_id;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            records = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(records, columns=columns)


remaining_news = get_remaining_news()

print(f"Noticias restantes: {len(remaining_news)}")

for _, news in remaining_news.iterrows():
    print(f"ID: {news['rss_news_id']}")
    print(f"Título: {news['title']}")
    print(f"Descripción: {news['description']}")
    print("-" * 80)

Noticias restantes: 116
ID: 1
Título: Guía para obtener 15 GB de espacio gratis en Google Drive, Gmail y Google Fotos
Descripción: Videos, respaldos antiguos e instaladores suelen estar entre los principales responsables, y al borrarlos y vaciar la papelera se recupera almacenamiento efectivo
--------------------------------------------------------------------------------
ID: 2
Título: Epic Games Store regala un nuevo juego de terror cooperativo para PC
Descripción: La tienda suma un indie de supervivencia y exploración en un bosque paranormal, con énfasis en comunicación en equipo y tensión atmosférica
--------------------------------------------------------------------------------
ID: 4
Título: WhatsApp refuerza la seguridad de las cuentas para evitar estafas ante llamadas de números desconocidos
Descripción: La nueva función busca combatir las tácticas de los estafadores, que suelen recurrir a llamadas inesperadas y mensajes urgentes para engañar a sus víctimas
---------------------

In [28]:
# Noticias relevantes identificadas en la segunda revisión.
remaining_relevant_ids = [26, 150]

# Noticias que requieren una revisión adicional.
remaining_uncertain_ids = [23, 87, 91, 157, 172, 186]

# Crear una copia independiente para la evaluación.
remaining_evaluation = remaining_news.copy()

# Inicializar como no relevantes.
remaining_evaluation["manual_label"] = 0

# Asignar las etiquetas relevantes.
remaining_evaluation.loc[
    remaining_evaluation["rss_news_id"].isin(remaining_relevant_ids),
    "manual_label",
] = 1

# Asignar las etiquetas dudosas.
remaining_evaluation.loc[
    remaining_evaluation["rss_news_id"].isin(remaining_uncertain_ids),
    "manual_label",
] = -1

# Comprobar que todos los identificadores estén presentes.
reviewed_ids = set(
    remaining_relevant_ids + remaining_uncertain_ids
)

assert reviewed_ids.issubset(
    set(remaining_evaluation["rss_news_id"])
)

print("Distribución de etiquetas:")
print(remaining_evaluation["manual_label"].value_counts().sort_index())

Distribución de etiquetas:
manual_label
-1      6
 0    108
 1      2
Name: count, dtype: int64


In [29]:
# Unificar las etiquetas de ambos conjuntos.
full_evaluation = pd.concat(
    [
        evaluation_dataset,
        remaining_evaluation,
    ],
    ignore_index=True,
)

# Validar integridad del conjunto de evaluación.
assert len(full_evaluation) == 186
assert full_evaluation["rss_news_id"].is_unique
assert full_evaluation["manual_label"].notna().all()

print("Total de noticias:", len(full_evaluation))

print("\nDistribución general:")
print(
    full_evaluation["manual_label"]
    .value_counts()
    .sort_index()
)

display(
    full_evaluation[
        ["rss_news_id", "title", "manual_label"]
    ].sort_values("rss_news_id")
)

Total de noticias: 186

Distribución general:
manual_label
-1     16
 0    137
 1     33
Name: count, dtype: int64


,rss_news_id,title,manual_label
70,1,Guía para obtener 15 GB de espacio gratis en G...,0
71,2,Epic Games Store regala un nuevo juego de terr...,0
0,3,Bill Gates advierte sobre la IA: “será el mayo...,1
72,4,WhatsApp refuerza la seguridad de las cuentas ...,0
73,5,Gratis en Steam: el juego que combina superviv...,0
...,...,...,...
181,182,Chubut: descubrieron un esqueleto casi complet...,0
182,183,La cúrcuma podría ser tan eficaz como el omepr...,0
183,184,Investigadores argentinos estudian la contamin...,0
184,185,"La Rioja presentó “Federal”, el primer aceite ...",0


In [30]:
# Actualizar etiquetas revisadas
reviewed_labels = {"21":-1,"23":1,"24":0,"35":-1,"87":1,"91":0,"93":1,"94":1,"95":0,"97":0,"111":0,"113":0,"135":0,"157":0,"172":0,"186":0}

full_evaluation["manual_label"] = full_evaluation["rss_news_id"].map(reviewed_labels).fillna(full_evaluation["manual_label"]).astype(int)

print(full_evaluation["manual_label"].value_counts().sort_index())

manual_label
-1     16
 0    137
 1     33
Name: count, dtype: int64


## Evaluación de relevancia temática

Se evaluará el modelo mDeBERTa-v3-base-mnli-xnli mediante
inferencia de lenguaje natural (NLI), utilizando como hipótesis
temática: "La noticia habla sobre inteligencia artificial".

El modelo procesará las 186 noticias recopiladas, independientemente
de si fueron seleccionadas por el filtro inicial de palabras clave.

Se utilizarán el título y la descripción de cada noticia como
texto de entrada, manteniendo las mismas condiciones de los
experimentos preliminares.

Las predicciones se contrastarán posteriormente con las etiquetas
de referencia para calcular las métricas de clasificación.

In [31]:
# Preparar el conjunto completo para la inferencia.
classification_dataset = full_evaluation.copy()

# Construir el texto utilizado como entrada del modelo.
classification_dataset["classification_text"] = (
    classification_dataset.apply(
        lambda row: build_news_text(
            row["title"],
            row["description"],
        ),
        axis=1,
    )
)

# Definir la hipótesis temática.
TOPIC_HYPOTHESIS = (
    "La noticia habla sobre inteligencia artificial."
)

print(f"Noticias para clasificar: {len(classification_dataset)}")

display(
    classification_dataset[
        [
            "rss_news_id",
            "classification_text",
            "manual_label",
        ]
    ].head()
)

Noticias para clasificar: 186


,rss_news_id,classification_text,manual_label
0,3,Título: Bill Gates advierte sobre la IA: “será...,1
1,7,Título: Time reveló las 100 personalidades más...,1
2,8,Título: Un informe reveló cómo 700 agentes de ...,1
3,12,Título: El costo de Bitcoin para este jueves\n...,0
4,13,Título: Bill Gates pide reservar empleos solo ...,1


### Inferencia del modelo de relevancia temática

Se aplicará el modelo mDeBERTa-v3-base-mnli-xnli sobre el conjunto
completo de noticias, utilizando el título y la descripción como
premisa y la hipótesis temática definida previamente.

Para cada noticia se obtendrán tres puntuaciones de inferencia
de lenguaje natural (NLI):

- Entailment: grado de respaldo de la hipótesis por el texto.
- Neutral: ausencia de evidencia concluyente.
- Contradiction: grado de contradicción de la hipótesis.

Las puntuaciones se conservarán para analizar la capacidad
discriminativa del modelo antes de establecer un umbral de
clasificación.

In [32]:
from tqdm.auto import tqdm

# Almacenar las puntuaciones obtenidas por el modelo.
classification_results = []

# Ejecutar la inferencia sobre todas las noticias.
for _, news in tqdm(
    classification_dataset.iterrows(),
    total=len(classification_dataset),
    desc="Clasificando noticias",
):
    scores = evaluate_topic_entailment(
        text=news["classification_text"],
        hypothesis=TOPIC_HYPOTHESIS,
        classifier=relevance_classifier,
    )

    classification_results.append({
        "rss_news_id": news["rss_news_id"],
        **scores,
    })

# Construir el DataFrame de resultados.
relevance_results = pd.DataFrame(classification_results)

print(f"Noticias clasificadas: {len(relevance_results)}")

display(relevance_results.head())

Clasificando noticias: 100%|██████████| 186/186 [00:29<00:00,  6.32it/s]

Noticias clasificadas: 186


,rss_news_id,entailment_score,neutral_score,contradiction_score
0,3,0.998671,0.001206,0.000123
1,7,0.997252,0.002625,0.000123
2,8,0.996118,0.003819,0.000063
3,12,0.000291,0.002454,0.997254
4,13,0.997712,0.002174,0.000114


### Análisis exploratorio de las puntuaciones NLI

Se integran las puntuaciones obtenidas mediante mDeBERTa
con las etiquetas de referencia asignadas durante la revisión
del conjunto de noticias.

El objetivo es analizar la distribución de las puntuaciones
de entailment para noticias relevantes y no relevantes,
identificando posibles solapamientos entre ambas clases.

Este análisis permitirá estudiar el comportamiento del
clasificador antes de establecer un criterio de decisión.

In [33]:
# Integrar las puntuaciones del modelo con las etiquetas.
evaluation_results = classification_dataset.merge(
    relevance_results,
    on="rss_news_id",
    how="inner",
    validate="one_to_one",
)

# Verificar la integridad de la integración.
assert len(evaluation_results) == len(classification_dataset)

# Excluir temporalmente los casos que sigan siendo dudosos.
validated_results = evaluation_results[
    evaluation_results["manual_label"].isin([0, 1])
].copy()

# Analizar las puntuaciones según la etiqueta de referencia.
score_summary = (
    validated_results.groupby("manual_label")["entailment_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

print("Distribución de puntuaciones por clase:")
display(score_summary)

Distribución de puntuaciones por clase:


,count,mean,median,min,max
manual_label,,,,,
0,137,0.0314,0.0003,0.0001,0.9896
1,33,0.6294,0.9923,0.0004,0.9988


In [34]:
# Definir un umbral provisional de clasificación.
RELEVANCE_THRESHOLD = 0.5

# Generar predicciones binarias.
validated_results["predicted_label"] = (
    validated_results["entailment_score"] >= RELEVANCE_THRESHOLD
).astype(int)

# Identificar discrepancias con las etiquetas de referencia.
classification_errors = validated_results[
    validated_results["manual_label"]
    != validated_results["predicted_label"]
].copy()

# Ordenar los errores por puntuación.
classification_errors = classification_errors.sort_values(
    "entailment_score",
    ascending=False,
)

print(f"Noticias evaluadas: {len(validated_results)}")
print(f"Posibles errores: {len(classification_errors)}")

display(
    classification_errors[
        [
            "rss_news_id",
            "title",
            "manual_label",
            "predicted_label",
            "entailment_score",
        ]
    ]
)

Noticias evaluadas: 170
Posibles errores: 16


,rss_news_id,title,manual_label,predicted_label,entailment_score
7,19,Propinas para robots: la polémica detrás de la...,0,1,0.989592
174,175,Especialistas argentinos crean un robot capaz ...,0,1,0.987576
17,46,Gleeden anuncia podcast sobre infidelidad y no...,0,1,0.975158
61,118,"Erling Haaland, el jugador noruego que la romp...",0,1,0.690108
69,136,Científicos argentinos desarrollan una vincha ...,1,0,0.392533
43,125,“La IA no depende de cómo la usamos sino de có...,1,0,0.037430
58,148,M-SCORE: Un Modelo Cuantitativo y Explicable d...,1,0,0.025279
151,150,Arquitectura Híbrida Interpretable basada en W...,1,0,0.006187
19,48,Así se verían Las Guerreras K-pop como persona...,1,0,0.002105
53,114,La Torre Eiffel derrumbada: Google Earth elimi...,1,0,0.002054


In [35]:
# Examinar el contenido utilizado para clasificar cada noticia.
for _, news in classification_errors.iterrows():
    print("=" * 90)
    print(f"ID: {news['rss_news_id']}")
    print(f"Etiqueta real: {news['manual_label']}")
    print(f"Predicción: {news['predicted_label']}")
    print(f"Entailment: {news['entailment_score']:.4f}")
    print()
    print(news["classification_text"])
    print()

ID: 19
Etiqueta real: 0
Predicción: 1
Entailment: 0.9896

Título: Propinas para robots: la polémica detrás de la automatización en bares y cafeterías
Descripción: La automatización en coctelería incorpora terminales que sugieren porcentajes aunque no haya servicio humano directo, y expertos advierten que la falta de reglas y transparencia deja dudas

ID: 175
Etiqueta real: 0
Predicción: 1
Entailment: 0.9876

Título: Especialistas argentinos crean un robot capaz de descubrir fallas en las tuberías de cualquier ciudad
Descripción: El equipo transmite en tiempo real, tiene una cámara que gira 360 grados y una luz LED que alumbra los lugares más oscuros. Además, está elaborado con un plástico que resiste los efectos corrosivos de los ácidos presentes en las cañerías.

ID: 46
Etiqueta real: 0
Predicción: 1
Entailment: 0.9752

Título: Gleeden anuncia podcast sobre infidelidad y no monogamia: el desafio tecnológico de las relaciones
Descripción: Gleeden es una plataforma de citas extraconyuga

### Experimento de sensibilidad a la formulación de hipótesis

Se evaluará la sensibilidad del modelo NLI ante diferentes
formulaciones de la hipótesis temática.

Se utilizarán las noticias identificadas como discrepancias
en la clasificación inicial, comparando sus puntuaciones
de entailment bajo tres formulaciones alternativas.

El objetivo es explorar si una hipótesis que contemple
explícitamente las aplicaciones de inteligencia artificial
permite mejorar la identificación de noticias relevantes.

Este experimento es exploratorio y no constituye una
evaluación independiente del rendimiento del modelo.

In [36]:
# Definir las hipótesis que se compararán.
topic_hypotheses = {
    "original": (
        "La noticia habla sobre inteligencia artificial."
    ),
    "aplicaciones": (
        "La noticia trata sobre sistemas de inteligencia "
        "artificial o sus aplicaciones."
    ),
    "especifica": (
        "La noticia trata sobre aprendizaje automático, "
        "modelos generativos o aplicaciones de inteligencia artificial."
    ),
}

# Recuperar los textos de los casos problemáticos.
experiment_news = classification_errors.copy()

# Evaluar cada formulación.
for hypothesis_name, hypothesis_text in topic_hypotheses.items():
    experiment_news[f"score_{hypothesis_name}"] = [
        evaluate_topic_entailment(
            text=news_text,
            hypothesis=hypothesis_text,
            classifier=relevance_classifier,
        )["entailment_score"]
        for news_text in experiment_news["classification_text"]
    ]

display(
    experiment_news[
        [
            "rss_news_id",
            "manual_label",
            "score_original",
            "score_aplicaciones",
            "score_especifica",
        ]
    ].round(4)
)

,rss_news_id,manual_label,score_original,score_aplicaciones,score_especifica
7,19,0,0.9896,0.9606,0.0133
174,175,0,0.9876,0.9810,0.0008
17,46,0,0.9752,0.0009,0.0002
61,118,0,0.6901,0.0093,0.0165
69,136,1,0.3925,0.1960,0.0002
43,125,1,0.0374,0.0109,0.0008
58,148,1,0.0253,0.1476,0.0004
151,150,1,0.0062,0.0077,0.0124
19,48,1,0.0021,0.0018,0.0009
53,114,1,0.0021,0.0016,0.0009


In [37]:
# Definir la hipótesis temática ampliada.
ROBUST_HYPOTHESIS = (
    "El tema central de esta noticia es la inteligencia artificial, "
    "su desarrollo, sus aplicaciones prácticas o sus consecuencias "
    "sociales, económicas y tecnológicas."
)

# Evaluar la nueva hipótesis sobre los casos problemáticos.
experiment_news["score_robust"] = [
    evaluate_topic_entailment(
        text=news_text,
        hypothesis=ROBUST_HYPOTHESIS,
        classifier=relevance_classifier,
    )["entailment_score"]
    for news_text in experiment_news["classification_text"]
]

# Comparar los resultados con la hipótesis original.
experiment_news["predicted_robust"] = (
    experiment_news["score_robust"] >= RELEVANCE_THRESHOLD
).astype(int)

display(
    experiment_news[
        [
            "rss_news_id",
            "manual_label",
            "score_original",
            "score_robust",
            "predicted_robust",
        ]
    ].round(4)
)

,rss_news_id,manual_label,score_original,score_robust,predicted_robust
7,19,0,0.9896,0.0008,0
174,175,0,0.9876,0.0008,0
17,46,0,0.9752,0.0015,0
61,118,0,0.6901,0.0006,0
69,136,1,0.3925,0.0006,0
43,125,1,0.0374,0.0005,0
58,148,1,0.0253,0.0002,0
151,150,1,0.0062,0.0009,0
19,48,1,0.0021,0.0016,0
53,114,1,0.0021,0.0004,0


### Experimento de clasificación mediante múltiples hipótesis temáticas

Se propone una estrategia de clasificación basada en múltiples
hipótesis especializadas, utilizando el modelo
mDeBERTa-v3-base-mnli-xnli.

En lugar de representar la relevancia temática mediante una
única afirmación, se consideran cinco dimensiones:

1. Inteligencia artificial en general.
2. Aprendizaje automático y aprendizaje profundo.
3. Inteligencia artificial generativa.
4. Aplicaciones prácticas de inteligencia artificial.
5. Impacto social, económico y regulatorio de la IA.

Para cada noticia se calculará la puntuación de entailment
correspondiente a cada hipótesis.

Como estrategia exploratoria de agregación, se utilizará
la puntuación máxima obtenida entre las cinco hipótesis.

Los resultados se compararán con la clasificación original
para analizar si mejora la identificación de noticias relevantes.

In [39]:
# Definir hipótesis robustas para cada dimensión temática.
ROBUST_SPECIALIZED_HYPOTHESES = {
    "general": (
        "El tema central de esta noticia es la inteligencia artificial, "
        "su investigación, desarrollo, avances tecnológicos o "
        "la creación de nuevos sistemas inteligentes."
    ),
    "machine_learning": (
        "El tema central de esta noticia es el aprendizaje automático, "
        "el aprendizaje profundo, las redes neuronales, los modelos "
        "predictivos o sus aplicaciones en problemas reales."
    ),
    "generative_ai": (
        "El tema central de esta noticia es la inteligencia artificial "
        "generativa, los modelos de lenguaje, los asistentes como "
        "ChatGPT o Gemini, la generación de imágenes, audio o texto "
        "y sus aplicaciones prácticas."
    ),
    "applications": (
        "El tema central de esta noticia es la aplicación de "
        "inteligencia artificial en ámbitos como la salud, "
        "la educación, las empresas, la industria, la ciencia "
        "o la automatización inteligente."
    ),
    "impact": (
        "El tema central de esta noticia es el impacto de la "
        "inteligencia artificial en la sociedad, la economía, "
        "el empleo, la educación, la privacidad, la seguridad, "
        "la ética o su regulación."
    ),
}

# Crear una copia de los casos problemáticos.
robust_multi_results = experiment_news.copy()

# Evaluar las cinco hipótesis robustas.
for hypothesis_name, hypothesis_text in (
    ROBUST_SPECIALIZED_HYPOTHESES.items()
):
    robust_multi_results[f"robust_{hypothesis_name}"] = [
        evaluate_topic_entailment(
            text=news_text,
            hypothesis=hypothesis_text,
            classifier=relevance_classifier,
        )["entailment_score"]
        for news_text in robust_multi_results["classification_text"]
    ]

# Obtener las columnas de puntuaciones.
robust_score_columns = [
    f"robust_{name}"
    for name in ROBUST_SPECIALIZED_HYPOTHESES
]

# Calcular la puntuación máxima.
robust_multi_results["max_robust_score"] = (
    robust_multi_results[robust_score_columns].max(axis=1)
)

# Generar la predicción binaria.
robust_multi_results["predicted_robust_multi"] = (
    robust_multi_results["max_robust_score"]
    >= RELEVANCE_THRESHOLD
).astype(int)

# Identificar la dimensión con mayor puntuación.
robust_multi_results["best_robust_hypothesis"] = (
    robust_multi_results[robust_score_columns]
    .idxmax(axis=1)
    .str.replace("robust_", "", regex=False)
)

# Comparar los resultados.
display(
    robust_multi_results[
        [
            "rss_news_id",
            "manual_label",
            "predicted_robust_multi",
            "max_robust_score",
            "best_robust_hypothesis",
        ]
    ].round(4)
)

,rss_news_id,manual_label,predicted_robust_multi,max_robust_score,best_robust_hypothesis
7,19,0,0,0.0031,generative_ai
174,175,0,0,0.0040,generative_ai
17,46,0,0,0.0046,generative_ai
61,118,0,0,0.0073,generative_ai
69,136,1,0,0.0011,generative_ai
43,125,1,0,0.0009,general
58,148,1,0,0.0005,impact
151,150,1,0,0.0055,generative_ai
19,48,1,1,0.8245,generative_ai
53,114,1,0,0.0016,generative_ai


### Evaluación global de la estrategia multihipótesis robusta

Se aplicará la configuración de cinco hipótesis especializadas
sobre el conjunto completo de 186 noticias.

Para cada noticia se conservará la máxima puntuación de
entailment y se generará una predicción binaria utilizando
un umbral provisional de 0,5.

Posteriormente, se compararán las predicciones con las etiquetas
de referencia disponibles, excluyendo los casos no resueltos.

Este análisis permitirá evaluar si las mejoras observadas
en los casos problemáticos se mantienen en el conjunto completo.

In [42]:
# Crear una copia del conjunto completo.
full_robust_results = classification_dataset.copy()

# Evaluar las hipótesis robustas sobre todas las noticias.
for hypothesis_name, hypothesis_text in (
    ROBUST_SPECIALIZED_HYPOTHESES.items()
):
    full_robust_results[f"robust_{hypothesis_name}"] = [
        evaluate_topic_entailment(
            text=news_text,
            hypothesis=hypothesis_text,
            classifier=relevance_classifier,
        )["entailment_score"]
        for news_text in full_robust_results["classification_text"]
    ]

# Calcular la puntuación máxima entre las cinco hipótesis.
full_robust_results["max_robust_score"] = (
    full_robust_results[robust_score_columns].max(axis=1)
)

# Generar las predicciones.
full_robust_results["predicted_robust_multi"] = (
    full_robust_results["max_robust_score"]
    >= RELEVANCE_THRESHOLD
).astype(int)

# Excluir las etiquetas dudosas de la evaluación.
full_robust_validated = full_robust_results[
    full_robust_results["manual_label"].isin([0, 1])
].copy()

# Calcular métricas comparables con el modelo original.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

comparison = []

for model_name, predictions in [
    (
        "NLI original",
        validated_results["predicted_label"],
    ),
    (
        "NLI multihipótesis robusto",
        full_robust_validated["predicted_robust_multi"],
    ),
]:
    # Alinear predicciones y etiquetas mediante el identificador.
    if model_name == "NLI original":
        results = validated_results
    else:
        results = full_robust_validated

    comparison.append({
        "estrategia": model_name,
        "accuracy": accuracy_score(
            results["manual_label"], predictions
        ),
        "precision": precision_score(
            results["manual_label"], predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            results["manual_label"], predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            results["manual_label"], predictions,
            zero_division=0,
        ),
    })

display(pd.DataFrame(comparison).round(4))

,estrategia,accuracy,precision,recall,f1
0,NLI original,0.9059,0.84,0.6364,0.7241
1,NLI multihipótesis robusto,0.8235,0.80,0.1212,0.2105


### Evaluación del clasificador posterior al filtrado por palabras clave

Se evalúa el rendimiento del modelo mDeBERTa sobre las noticias
preseleccionadas mediante palabras clave.

Se comparan dos estrategias:

1. Keywords: todas las noticias candidatas se consideran relevantes.
2. Keywords + mDeBERTa: las noticias candidatas se clasifican
   mediante la puntuación de entailment y un umbral de 0,5.

Se calculan precision, recall y F1-score sobre las noticias
con etiquetas de referencia binarias.

Adicionalmente, se mide la recuperación de noticias relevantes
del sistema completo respecto del conjunto original.

In [44]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Identificar las noticias seleccionadas por keywords.
candidate_ids = set(candidate_news["rss_news_id"])

# Recuperar los resultados NLI de las noticias candidatas.
candidate_evaluation = evaluation_results[
    evaluation_results["rss_news_id"].isin(candidate_ids)
    & evaluation_results["manual_label"].isin([0, 1])
].copy()

# Predicción del filtro de keywords: todas son candidatas.
candidate_evaluation["predicted_keywords"] = 1

# Predicción del clasificador mDeBERTa original.
candidate_evaluation["predicted_nli"] = (
    candidate_evaluation["entailment_score"] >= RELEVANCE_THRESHOLD
).astype(int)

# Calcular las métricas de ambas estrategias.
comparison = []

for strategy, prediction_column in [
    ("Solo keywords", "predicted_keywords"),
    ("Keywords + mDeBERTa", "predicted_nli"),
]:
    y_true = candidate_evaluation["manual_label"]
    y_pred = candidate_evaluation[prediction_column]

    comparison.append({
        "estrategia": strategy,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "noticias_seleccionadas": int(y_pred.sum()),
    })

comparison_df = pd.DataFrame(comparison)

print(f"Noticias candidatas: {len(candidate_ids)}")
print(f"Noticias evaluadas: {len(candidate_evaluation)}")

display(comparison_df.round(4))

Noticias candidatas: 70
Noticias evaluadas: 60


,estrategia,accuracy,precision,recall,f1,noticias_seleccionadas
0,Solo keywords,0.5167,0.5167,1.0000,0.6813,60
1,Keywords + mDeBERTa,0.7833,0.8750,0.6774,0.7636,24


### Análisis de recuperación semántica fuera del filtro de keywords

Se analizarán las noticias que no fueron seleccionadas mediante
el filtrado inicial por palabras clave.

El objetivo es identificar noticias potencialmente relevantes
mediante las puntuaciones NLI, incluso cuando no presentan
coincidencias con las palabras clave configuradas.

Se conservarán las puntuaciones continuas y se utilizará
un umbral de 0,5 únicamente como criterio exploratorio
para identificar noticias candidatas a revisión.

Este análisis permitirá estudiar la capacidad del modelo
para complementar la recuperación basada en keywords.

In [45]:
# Identificar noticias que no pasaron el filtro de keywords.
non_candidate_news = evaluation_results[
    ~evaluation_results["rss_news_id"].isin(candidate_ids)
].copy()

# Ordenar por relevancia semántica descendente.
non_candidate_news = non_candidate_news.sort_values(
    "entailment_score",
    ascending=False,
)

# Identificar noticias potencialmente recuperadas.
recovered_news = non_candidate_news[
    non_candidate_news["entailment_score"] >= RELEVANCE_THRESHOLD
].copy()

print(f"Noticias sin keywords: {len(non_candidate_news)}")
print(f"Noticias recuperadas por NLI: {len(recovered_news)}")

# Mostrar las noticias recuperadas.
display(
    recovered_news[
        [
            "rss_news_id",
            "title",
            "manual_label",
            "entailment_score",
        ]
    ].round(4)
)

# Verificar los casos relevantes conocidos.
display(
    non_candidate_news[
        non_candidate_news["manual_label"] == 1
    ][
        [
            "rss_news_id",
            "title",
            "entailment_score",
        ]
    ].round(4)
)

Noticias sin keywords: 116
Noticias recuperadas por NLI: 4


,rss_news_id,title,manual_label,entailment_score
124,87,Waymo lleva sus robotaxis a Múnich y prepara s...,-1,0.9953
83,23,Por qué Londres está frenando la llegada de lo...,-1,0.9912
174,175,Especialistas argentinos crean un robot capaz ...,0,0.9876
185,186,A lo Terminator: científicos británicos crearo...,-1,0.9289


,rss_news_id,title,entailment_score
151,150,Arquitectura Híbrida Interpretable basada en W...,0.0062
85,26,Google lanzó Gemini 3.5 Transcribe: pasa más d...,0.0011


### Clasificación de relevancia mediante señales léxicas y semánticas

Se utiliza el modelo mDeBERTa con una hipótesis NLI robusta
para estimar la relevancia temática de las noticias.

Posteriormente, se combina la puntuación semántica con
las coincidencias de palabras clave para generar cuatro
categorías:

- Alta relevancia: coincidencia de keywords y NLI positivo.
- Revisar (keywords): coincidencia de keywords, NLI negativo.
- Revisar (NLI): sin keywords, pero NLI positivo.
- Baja relevancia: ninguna de las dos señales es positiva.

Las categorías constituyen estimaciones de relevancia,
no decisiones definitivas de exclusión.

In [48]:
import torch
from transformers import pipeline

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

classifier = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=-1,  # CPU
)

# Evitar problemas de precisión FP16 en CPU.
classifier.model.float()

print("Modelo mDeBERTa cargado correctamente.")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 919.55it/s]


Modelo mDeBERTa cargado correctamente.


In [57]:
NLI_HYPOTHESES = {
    "original": (
        "La noticia habla sobre inteligencia artificial."
    ),
    "general": (
        "El tema central de esta noticia es la inteligencia artificial."
    ),
    "applications": (
        "La noticia trata sobre el desarrollo y las aplicaciones "
        "prácticas de la inteligencia artificial."
    ),
    "impact": (
        "La noticia trata sobre las consecuencias sociales, "
        "económicas y tecnológicas de la inteligencia artificial."
    ),
}

RELEVANCE_THRESHOLD = 0.5

In [58]:
# Crear el dataset que almacenará todas las evaluaciones.
nli_results = classification_dataset.copy()

# Incorporar las etiquetas de referencia, evitando duplicados.
if "manual_label" not in nli_results.columns:
    nli_results = nli_results.merge(
        full_evaluation[["rss_news_id", "manual_label"]],
        on="rss_news_id",
        how="left",
        validate="one_to_one",
    )

# Incorporar el resultado del filtrado por keywords.
nli_results["keyword_match"] = (
    nli_results["rss_news_id"].isin(candidate_ids)
)

# Almacenar las métricas de cada evaluación.
nli_metrics = []

print(f"Noticias disponibles: {len(nli_results)}")
print(
    "Noticias con etiquetas binarias:",
    nli_results["manual_label"].isin([0, 1]).sum(),
)

Noticias disponibles: 186
Noticias con etiquetas binarias: 170


In [59]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def evaluate_nli_hypothesis(
    hypothesis_name,
    hypothesis_text,
    dataset,
    classifier,
    threshold=0.5,
):
    """
    Evalúa una hipótesis NLI de forma independiente.

    Agrega al dataset una columna con la puntuación de entailment
    y calcula las métricas sobre las noticias con etiquetas binarias.
    """
    score_column = f"nli_{hypothesis_name}_score"
    prediction_column = f"nli_{hypothesis_name}_prediction"

    scores = []

    for text in dataset["classification_text"]:
        result = evaluate_topic_entailment(
            text,
            hypothesis_text,
            classifier,
        )

        scores.append(result["entailment_score"])

    # Conservar la puntuación continua.
    dataset[score_column] = scores

    # Predicción binaria utilizada únicamente para la evaluación.
    dataset[prediction_column] = (
        dataset[score_column] >= threshold
    ).astype(int)

    # Seleccionar noticias con etiquetas de referencia binarias.
    validated = dataset[
        dataset["manual_label"].isin([0, 1])
    ].copy()

    y_true = validated["manual_label"].astype(int)
    y_pred = validated[prediction_column]

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    metrics = {
        "hipotesis": hypothesis_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }

    print(f"Hipótesis evaluada: {hypothesis_name}")

    display(
        pd.DataFrame([metrics]).round(4)
    )

    return metrics

In [60]:
metrics_original = evaluate_nli_hypothesis(
    hypothesis_name="original",
    hypothesis_text=NLI_HYPOTHESES["original"],
    dataset=nli_results,
    classifier=classifier,
    threshold=RELEVANCE_THRESHOLD,
)

nli_metrics.append(metrics_original)

Hipótesis evaluada: original


,hipotesis,accuracy,precision,recall,f1,TP,FP,FN,TN
0,original,0.9059,0.84,0.6364,0.7241,21,4,12,133


In [61]:
metrics_general = evaluate_nli_hypothesis(
    hypothesis_name="general",
    hypothesis_text=NLI_HYPOTHESES["general"],
    dataset=nli_results,
    classifier=classifier,
    threshold=RELEVANCE_THRESHOLD,
)

nli_metrics.append(metrics_general)

Hipótesis evaluada: general


,hipotesis,accuracy,precision,recall,f1,TP,FP,FN,TN
0,general,0.8176,1.0,0.0606,0.1143,2,0,31,137


In [62]:
metrics_applications = evaluate_nli_hypothesis(
    hypothesis_name="applications",
    hypothesis_text=NLI_HYPOTHESES["applications"],
    dataset=nli_results,
    classifier=classifier,
    threshold=RELEVANCE_THRESHOLD,
)

nli_metrics.append(metrics_applications)

Hipótesis evaluada: applications


,hipotesis,accuracy,precision,recall,f1,TP,FP,FN,TN
0,applications,0.8471,0.8889,0.2424,0.381,8,1,25,136


In [63]:
metrics_impact = evaluate_nli_hypothesis(
    hypothesis_name="impact",
    hypothesis_text=NLI_HYPOTHESES["impact"],
    dataset=nli_results,
    classifier=classifier,
    threshold=RELEVANCE_THRESHOLD,
)

nli_metrics.append(metrics_impact)

Hipótesis evaluada: impact


,hipotesis,accuracy,precision,recall,f1,TP,FP,FN,TN
0,impact,0.8176,1.0,0.0606,0.1143,2,0,31,137


In [64]:
metrics_comparison = pd.DataFrame(nli_metrics)

display(
    metrics_comparison.sort_values(
        "f1",
        ascending=False,
    ).round(4)
)

,hipotesis,accuracy,precision,recall,f1,TP,FP,FN,TN
0,original,0.9059,0.8400,0.6364,0.7241,21,4,12,133
2,applications,0.8471,0.8889,0.2424,0.3810,8,1,25,136
1,general,0.8176,1.0000,0.0606,0.1143,2,0,31,137
3,impact,0.8176,1.0000,0.0606,0.1143,2,0,31,137


In [65]:
score_columns = [
    f"nli_{name}_score"
    for name in NLI_HYPOTHESES
]

display(
    nli_results[
        [
            "rss_news_id",
            "title",
            "manual_label",
            "keyword_match",
            *score_columns,
        ]
    ].round(4)
)

,rss_news_id,title,manual_label,keyword_match,nli_original_score,nli_general_score,nli_applications_score,nli_impact_score
0,3,Bill Gates advierte sobre la IA: “será el mayo...,1,True,0.9987,0.0046,0.0206,0.9729
1,7,Time reveló las 100 personalidades más influye...,1,True,0.9973,0.0073,0.0004,0.0005
2,8,Un informe reveló cómo 700 agentes de IA de Op...,1,True,0.9961,0.0013,0.0665,0.0019
3,12,El costo de Bitcoin para este jueves,0,True,0.0003,0.0002,0.0003,0.0016
4,13,Bill Gates pide reservar empleos solo para hum...,1,True,0.9977,0.0005,0.0024,0.0007
...,...,...,...,...,...,...,...,...
181,182,Chubut: descubrieron un esqueleto casi complet...,0,False,0.0002,0.0002,0.0008,0.0031
182,183,La cúrcuma podría ser tan eficaz como el omepr...,0,False,0.0001,0.0001,0.0002,0.0002
183,184,Investigadores argentinos estudian la contamin...,0,False,0.0002,0.0001,0.0002,0.0002
184,185,"La Rioja presentó “Federal”, el primer aceite ...",0,False,0.0001,0.0001,0.0002,0.0002


In [66]:
# Columnas de predicción de las cuatro hipótesis.
prediction_columns = [
    f"nli_{name}_prediction"
    for name in NLI_HYPOTHESES
]

# Unión de las cuatro hipótesis NLI.
# Una noticia es detectada si al menos una hipótesis resulta positiva.
nli_results["nli_union"] = (
    nli_results[prediction_columns].any(axis=1)
)

# Unión de keywords y las cuatro hipótesis NLI.
nli_results["hybrid_union"] = (
    nli_results["keyword_match"]
    | nli_results["nli_union"]
)

# Conservar noticias con etiquetas binarias.
validated = nli_results[
    nli_results["manual_label"].isin([0, 1])
].copy()

# Estrategias que vamos a comparar.
strategies = {
    "Solo keywords": "keyword_match",
    "NLI original": "nli_original_prediction",
    "Unión 4 hipótesis NLI": "nli_union",
    "Keywords + NLI original": None,
    "Keywords + 4 hipótesis": "hybrid_union",
}

validated["keywords_original_union"] = (
    validated["keyword_match"]
    | validated["nli_original_prediction"].astype(bool)
)

strategies["Keywords + NLI original"] = "keywords_original_union"

# Calcular métricas de recuperación.
recovery_metrics = []

for strategy_name, prediction_column in strategies.items():
    y_true = validated["manual_label"].astype(int)
    y_pred = validated[prediction_column].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    recovery_metrics.append({
        "estrategia": strategy_name,
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "TP": int(tp),
        "FN": int(fn),
        "FP": int(fp),
        "TN": int(tn),
        "seleccionadas": int(y_pred.sum()),
    })

recovery_comparison = pd.DataFrame(recovery_metrics)

display(
    recovery_comparison.round(4)
)

,estrategia,recall,precision,f1,TP,FN,FP,TN,seleccionadas
0,Solo keywords,0.9394,0.5167,0.6667,31,2,29,108,60
1,NLI original,0.6364,0.8400,0.7241,21,12,4,133,25
2,Unión 4 hipótesis NLI,0.6364,0.8400,0.7241,21,12,4,133,25
3,Keywords + NLI original,0.9394,0.5082,0.6596,31,2,30,107,61
4,Keywords + 4 hipótesis,0.9394,0.5082,0.6596,31,2,30,107,61


In [67]:
# Noticias relevantes no detectadas por keywords ni por NLI.
false_negatives = validated[
    (validated["manual_label"] == 1)
    & (~validated["hybrid_union"])
].copy()

print(
    "Noticias relevantes no detectadas:",
    len(false_negatives),
)

display(
    false_negatives[
        [
            "rss_news_id",
            "title",
            "keyword_match",
            "nli_original_score",
            "nli_general_score",
            "nli_applications_score",
            "nli_impact_score",
        ]
    ].round(4)
)

Noticias relevantes no detectadas: 2


,rss_news_id,title,keyword_match,nli_original_score,nli_general_score,nli_applications_score,nli_impact_score
85,26,Google lanzó Gemini 3.5 Transcribe: pasa más d...,False,0.0011,0.0004,0.0018,0.0005
151,150,Arquitectura Híbrida Interpretable basada en W...,False,0.0062,0.0005,0.0035,0.0012


In [68]:
# Noticias recuperadas exclusivamente por las hipótesis adicionales.
additional_recovery = validated[
    (~validated["keyword_match"])
    & (~validated["nli_original_prediction"].astype(bool))
    & (
        validated[
            [
                "nli_general_prediction",
                "nli_applications_prediction",
                "nli_impact_prediction",
            ]
        ].any(axis=1)
    )
].copy()

print(
    "Noticias adicionales recuperadas:",
    len(additional_recovery),
)

print(
    "De ellas, relevantes:",
    (additional_recovery["manual_label"] == 1).sum(),
)

display(
    additional_recovery[
        [
            "rss_news_id",
            "title",
            "manual_label",
            "nli_general_score",
            "nli_applications_score",
            "nli_impact_score",
        ]
    ].round(4)
)

Noticias adicionales recuperadas: 0
De ellas, relevantes: 0


,rss_news_id,title,manual_label,nli_general_score,nli_applications_score,nli_impact_score
